In [ ]:
import numpy as np
from openai import OpenAI
from collections import namedtuple
import traceback

In [ ]:
from openai import OpenAI
client = OpenAI(
    api_key="sk-proj"
    )

In [ ]:
def fetch_log_probs_openai(text: str, top_probs=6, max_tokens=1):
    response = client.completions.create(
        model="gpt-4o-mini",
        prompt= f'{text}',
        max_tokens=max_tokens,
        logprobs=top_probs,
        temperature=1,
    )
    return response.choices[0].logprobs.top_logprobs[0]

In [ ]:
def dummy_fetch(text: str):
  return {'lol_one': 1, 'lol_two': 2, 'lol_three': 3}

In [ ]:
def check_beam(token_path: list, max_length: int):
  '''
  Метод получает на вход список токенов луча и возвращает True/False в зависимости
  от того, нашелся ли в тексте токена символ сепаратора
  True  если нашелся сепаратор или длина луча (в токенах) превысила установленное значение в глубину
  '''
  if len(token_path) >= max_length:
    return 'too_long'
  elif any(punct in ''.join(token_path[1:]) for punct in {'.', ',', '»', '«', '!', '?', ' ', '\n', '-', '"', ':', ';', '(', ')'}):
    return 'sep_found'
  return False


In [ ]:
def generate_beams_openai(initial_context: str, max_length=8):
    BeamState = namedtuple('BeamState', ['current_text', 'token_path', 'score_trace', 'finished'])

    def expand_beam(beam_state):
        """Expand a single beam by getting top next tokens"""
        current_text, token_path, score_trace, finished = beam_state

        # Проверяем, не нужно ли остановить бим
        if len(token_path) <= 1: #именно <= потому что первому токену можно быть с сепаратором + в функции check_beam список должен быть длиннее одного элемента, иначе сломается
          pass
        elif (check_result := check_beam(token_path, max_length)):
          print(f'остановили beam, причина: {check_result}, получилось {''.join(token_path)}')
          return [BeamState(current_text, token_path, score_trace, check_result)]

        try:
          # print('Сейчас буду доставать токен для контекста: ', current_text)
          log_probs = fetch_log_probs_openai(current_text)
          # log_probs = dummy_fetch(current_text)
          next_tokens = sorted(log_probs.items(), key=lambda x: x[1], reverse=True)#TODO: убрать лишнюю сортировку, получаю уже сортированное (поменять либо итерацию ниже, либо функцию фетч)
          new_beams = []

          for token, prob in next_tokens:
              new_text = f"{current_text}{token}"
              new_path = token_path + [token]
              new_score_trace = score_trace + [prob]  # accumulate probability
              new_beams.append(BeamState(new_text, new_path, new_score_trace, False))
          return new_beams

        except Exception as e:
          print(f"Ошибка при генерации: {e}")
          traceback.print_exc()
          return [beam_state]

    # Start with initial beam
    active_beams = [BeamState(initial_context, [], [], False)]
    completed_beams = []

    # Expand beams until all are completed
    while active_beams:
        beam = active_beams.pop(0)
        expanded = expand_beam(beam)

        for new_beam in expanded:
            current_text, token_path, score_trace, finished = new_beam
            if finished:
              completed_beams.append(new_beam)

            else:
              active_beams.append(new_beam)

    return completed_beams

In [ ]:
res = generate_beams_openai(initial_context='В тот момент') # WARNING: контекст должен быть БЕЗ пробела в конце. С ним модель начинает отвечать числами, получаются очень разные ответы

остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: sep_found
остановили beam, причина: se

In [ ]:
len(res)

[BeamState(current_text='Я уеду жить в Л.А', token_path=[' Л', '.А'], score_trace=[-0.08769222348928452, -9.876118659973145], finished='sep_found'),
 BeamState(current_text='Я уеду жить в другой город', token_path=[' другой', ' город'], score_trace=[-3.2126922607421875, -0.06884494423866272], finished='sep_found'),
 BeamState(current_text='Я уеду жить в другой край', token_path=[' другой', ' край'], score_trace=[-3.2126922607421875, -3.81884503364563], finished='sep_found'),
 BeamState(current_text='Я уеду жить в другой мир', token_path=[' другой', ' мир'], score_trace=[-3.2126922607421875, -4.068844795227051], finished='sep_found'),
 BeamState(current_text='Я уеду жить в другой дом', token_path=[' другой', ' дом'], score_trace=[-3.2126922607421875, -4.693844795227051], finished='sep_found'),
 BeamState(current_text='Я уеду жить в другой штат', token_path=[' другой', ' штат'], score_trace=[-3.2126922607421875, -6.068844795227051], finished='sep_found'),
 BeamState(current_text='Я уеду 

In [ ]:
left_contexts = ["В тот момент <mask>", # все наши контектсы
"Клиенты воровали из ресторана <mask>",
"Ему удалось вскрыть банку об острый край <mask>",
"Ей никак не суметь <mask>",
"Старуха была страшной -- <mask>",
"Выбирая вязаную шапочку, знайте, что лучше шапка цвета <mask>",
"У моего отца был счёт в швейцарском <mask>",
"Очень хочется заплести <mask>",
"В котёл бросают куски <mask>",
"На запись голоса <mask>",
"Ее сын Гриша умер <mask>",
"Можно будет <mask>",
"Музыканты играли на похоронах, разгружали <mask>",
"И не надо ставить это целью <mask>",
"Дрозды и сковрцы начали <mask>",
"Что может сделать самый сильный <mask>",
"Здесь потребуется <mask>",
"Какие главные лекарства должны <mask>",
"Применение микросхемы <mask>",
"Выходя замуж, ты надеялась обрести спокойствие, уютный <mask>",
"С нескрываемой <mask>",
"Однако здесь <mask>",
"Тому, кто <mask>",
"В качестве примера приводится <mask>",
"Он признаёт право каждого <mask>",
"Вспоминая <mask>",
"Он вскрыл пачку сухарей, <mask>",
"Но четыре года я не мог себя <mask>",
"На Ольгу Васильевну было написано <mask>",
"Я знал: их особенно <mask>",
"Очень тогда <mask>",
"Создать настоящие шедевры вам помогут <mask>",
"Наши власти позволяют себе <mask>",
"У нас в Волгограде многие придерживаются <mask>",
"Во избежание ожогов надо нанести на лицо небольшое <mask>",
"В вопросе послышался упрёк <mask>",
"За углом ― Морской музей, с бесчисленными моделями <mask>",
"Мне нравится сын коллеги, <mask>",
"Душа требовала <mask>",
"Наше правительство сделало <mask>",
"А промывать манную <mask>",
"Каждое утро на самый верх <mask>",
"Убедительно просим вас разборчиво заполнять <mask>",
"На болотах оставался ещё <mask>",
"Дорога ведет в глухой <mask>",
"На ведущей вниз <mask>",
"Когда она в самолёте <mask>",
"Возможности этих перемен будут обсуждаться в Париже <mask>",
"Там, недалеко от кухонной двери, сидел <mask>",
"В багажнике были лопата, <mask>",
"Врач прописал заживляющую <mask>",
"В бассейне <mask>",
"В резервациях <mask>",
"Мама брала меня с собой, и мы, сдав <mask>",
"Приблизительно в центре тайги <mask>",
"Не поручайте <mask>",
"В деревнях по-прежнему <mask>",
"В числе возможных кандидатов <mask>",
"Твоё тело расслабляется, и исчезает <mask>",
"Он был очень <mask>",
"Они не ели целый день, <mask>",
"В речи учёного прозвучало <mask>",
"Тема эта в то время была <mask>",
"Существует легенда, что <mask>",
"Он ловко поддел концом <mask>",
"У директора школы был тонкий <mask>",
"У Пашки <mask>",
"В конверте вместе с деньгами была <mask>",
"Стала стабильнее экономическая и политическая <mask>",
"В современном <mask>",
"Педагог предъявляет <mask>",
"Я сказал, что русский солдат <mask>",
"Старый шкаф <mask>",
"В сюжете этого фильма какие-то <mask>",
"Ирине досталась <mask>",
"На газовой плите стояла <mask>",
"Я люблю салат из картошки с зеленью, заправленный <mask>",
"Ненужный коврик из твёрдой <mask>",
"Что ты хочешь чтобы тебе <mask>",
"Журналист взял карандаш, <mask>",
"У мамы есть <mask>",
"Отвернув цветастое <mask>",
"У тебя впереди замечательный день, <mask>",
"Этот студент <mask>",
"Она почти не изменилась, только слегка <mask>",
"Перед ним снова была <mask>",
"Торговля продуктами питания является одной из самых <mask>",
"Когда родители <mask>",
"Товарищ генерал, <mask>",
"Считается, что коллекционирование <mask>",
"Телята быстро <mask>",
"Один футболист, который получил <mask>",
"Судя по огромному <mask>",
"Город, раскинувшийся вдоль <mask>",
"Под рукавом рубашки виднелся тонкий <mask>",
"На вторичном рынке жилья <mask>",
"Ваня раскрыл было <mask>",
"В мои обязанности входило утром включить <mask>",
"И на берегу озера тогда появляются <mask>",
"Наживка, на которую он ловил <mask>",
"Не обнаружив ничего в досье, сыщики решили <mask>",
"Когда мне хотелось <mask>",
"Взяв с собой фотоаппарат, вся <mask>",
"Собаку, виновницу случившегося, приказали <mask>",
"Мне было лень идти на стоянку и сметать <mask>",
"От смерти его спасла <mask>",
"По воскресеньям музыканты, исполнявшие <mask>",
"Он умел из любого <mask>",
"Если я еще увижу здесь хоть <mask>",
"Количество денег в обороте выросло благодаря <mask>",
"Он стал плохо <mask>",
"Работы выполняет <mask>",
"Зачем ему звонить, если откликается <mask>",
"Я слезал, щупал <mask>",
"Шею Лизы украшало ожерелье <mask>",
"Этот роман захватывает читателя с первой <mask>",
"Автор принадлежит к числу последних свидетелей <mask>",
"В темноте Иван задел острый <mask>",
"Приготовь себе диетические овощные блюда, <mask>",
"Олень бродил среди берёз, жевал <mask>",
"Причиной аварии был мобильный <mask>",
"После завершения <mask>",
"Чтобы придать объем <mask>",
"В лесу ветром <mask>",
"В каждом <mask>",
"Власть судов была такой <mask>",
"Володя каким-то образом <mask>",
"За два года накопилась <mask>",
"Если мы позволим этим людям <mask>",
"Думаю, большой <mask>",
"Елена сидела в кресле, молодая Мурка <mask>",
"От внимания наблюдателя не должна <mask>",
"Государством предлагается <mask>",
"Сделав мне знак помолчать, он приложил <mask>",
"Она успевала убраться, разморозить <mask>",
"Что используют для этой прически, <mask>",
"Она с досадой <mask>",
"Зоопарк ― это кусочек другого мира, находящийся в самом <mask>",
"Я сделала <mask>",
"За министром труда тянется целый <mask>",
"Мы установили камеру на новый <mask>",
"Ей хотелось выплеснуть чай на бежевый <mask>",
"На привале у озера <mask>",
"Покуда я нахожусь у власти, я буду предметом <mask>"]

In [ ]:
clean_contexts = [s.replace(" <mask>", "").replace("<mask>", "") for s in left_contexts]

In [ ]:
print(clean_contexts) # вроде бы нигде лишних пробелов не осталось

['В тот момент', 'Клиенты воровали из ресторана', 'Ему удалось вскрыть банку об острый край', 'Ей никак не суметь', 'Старуха была страшной --', 'Выбирая вязаную шапочку, знайте, что лучше шапка цвета', 'У моего отца был счёт в швейцарском', 'Очень хочется заплести', 'В котёл бросают куски', 'На запись голоса', 'Ее сын Гриша умер', 'Можно будет', 'Музыканты играли на похоронах, разгружали', 'И не надо ставить это целью', 'Дрозды и сковрцы начали', 'Что может сделать самый сильный', 'Здесь потребуется', 'Какие главные лекарства должны', 'Применение микросхемы', 'Выходя замуж, ты надеялась обрести спокойствие, уютный', 'С нескрываемой', 'Однако здесь', 'Тому, кто', 'В качестве примера приводится', 'Он признаёт право каждого', 'Вспоминая', 'Он вскрыл пачку сухарей,', 'Но четыре года я не мог себя', 'На Ольгу Васильевну было написано', 'Я знал: их особенно', 'Очень тогда', 'Создать настоящие шедевры вам помогут', 'Наши власти позволяют себе', 'У нас в Волгограде многие придерживаются', 'Во 

In [ ]:
import csv
import os
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
SAVE_DIR = Path("/content/drive/MyDrive/предсказания api 2 декабря")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Сохраняем файлы в:", SAVE_DIR)

Сохраняем файлы в: /content/drive/MyDrive/предсказания api 2 декабря


In [ ]:
def save_beams_context_csv(beams, context_text, index):
    """
    результаты бимов в гугл папку
    каждый контекст -- один файл
    """
    rows = []

    for beam in beams:
        rows.append({
            "text_test": context_text,  # изначальный контекст
            "text_word": beam.current_text,
            "token_path": beam.token_path,
            "score_trace": beam.score_trace,
            "finished": beam.finished
        })

    df = pd.DataFrame(rows)

    if len(df) > 0:
        df['token_path'] = df['token_path'].astype('object')
        df['score_trace'] = df['score_trace'].astype('object')

    name = "".join(c if c.isalnum() else "_" for c in context_text)[:40]
    filename = f"{name}.csv"

    filepath = SAVE_DIR / filename
    df.to_csv(filepath, index=False, encoding="utf-8")

    print("Сохранено в файл:", filepath)
    return df

In [ ]:
all_results = []

for idx, context in enumerate(clean_contexts, start=1):
    print(f"\nОбрабатываем контекст {idx}: {context!r}")

    beams = generate_beams_openai(context)

    saved_path = save_beams_context_csv(beams, context, idx)
    all_results.append({
        "context_index": idx,
        "context": context,
        "file": str(saved_path) if saved_path else None
    })

Выходные данные были обрезаны до нескольких последних строк (5000).
картой был
остановили beam, причина: sep_found, получилось 
картой.

остановили beam, причина: sep_found, получилось 
картой в
остановили beam, причина: sep_found, получилось 
картой не
остановили beam, причина: sep_found, получилось 
картой было
остановили beam, причина: sep_found, получилось 
картой 
остановили beam, причина: sep_found, получилось 
картой можно
остановили beam, причина: sep_found, получилось 
картой у
остановили beam, причина: sep_found, получилось 
картой я
остановили beam, причина: sep_found, получилось 
картой с
остановили beam, причина: sep_found, получилось 
картой и
остановили beam, причина: sep_found, получилось 
картой «
остановили beam, причина: sep_found, получилось 
картой по
остановили beam, причина: sep_found, получилось 
картой.
остановили beam, причина: sep_found, получилось 
картой управления
остановили beam, причина: sep_found, получилось 
картограф

остановили beam, причина: sep_fou

KeyboardInterrupt: 

# воскресенье

In [ ]:
res = generate_beams_openai(initial_context='В тот момент')


остановили beam, причина: sep_found, получилось , когда
остановили beam, причина: sep_found, получилось , как
остановили beam, причина: sep_found, получилось , пока
остановили beam, причина: sep_found, получилось , я
остановили beam, причина: sep_found, получилось , в
остановили beam, причина: sep_found, получилось , что
остановили beam, причина: sep_found, получилось  я не
остановили beam, причина: sep_found, получилось  я понял
остановили beam, причина: sep_found, получилось  я был
остановили beam, причина: sep_found, получилось  я почув
остановили beam, причина: sep_found, получилось  я пон
остановили beam, причина: sep_found, получилось  я была
остановили beam, причина: sep_found, получилось  он не
остановили beam, причина: sep_found, получилось  он был
остановили beam, причина: sep_found, получилось  он,
остановили beam, причина: sep_found, получилось  он почув
остановили beam, причина: sep_found, получилось  он понял
остановили beam, причина: sep_found, получилось  он уже
останов

In [ ]:
SAVE_DIR = Path("/content/drive/MyDrive/предсказания api 2 декабря")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Сохраняем файлы в:", SAVE_DIR)

Сохраняем файлы в: /content/drive/MyDrive/предсказания api 2 декабря


In [ ]:
import pandas as pd

In [ ]:
def save_beams_context_csv(beams, context_text, index):
    """
    результаты бимов в гугл папку
    каждый контекст -- один файл
    """
    rows = []

    for beam in beams:
        rows.append({
            "text_test": context_text,  # изначальный контекст
            "text_word": beam.current_text,
            "token_path": beam.token_path,
            "score_trace": beam.score_trace,
            "finished": beam.finished
        })

    df = pd.DataFrame(rows)

    if len(df) > 0:
        df['token_path'] = df['token_path'].astype('object')
        df['score_trace'] = df['score_trace'].astype('object')

    name = "".join(c if c.isalnum() else "_" for c in context_text)[:40]
    filename = f"{name}.csv"

    filepath = SAVE_DIR / filename
    df.to_csv(filepath, index=False, encoding="utf-8")

    print("Сохранено в файл:", filepath)
    return df



df_saved = save_beams_context_csv(res, 'В тот момент', index=0)


print(f"Количество результатов в res: {len(res)}")
print(f"Количество строк в сохраненном DataFrame: {len(df_saved)}")

Сохранено в файл: /content/drive/MyDrive/предсказания api 2 декабря/В_тот_момент.csv
Количество результатов в res: 1236
Количество строк в сохраненном DataFrame: 1236
